# NHANES 2021–2023 — Bước 2: Data Understanding

**Dự án:** Multi-Agent AI System for Diabetes Risk Assessment  
**Giai đoạn:** 3 — Bước 2/8  
**Người chạy:** Bạn (trên máy cá nhân)  
**Notebook:** Hướng dẫn **chưa chạy**, không chứa số liệu thực nghiệm, không huấn luyện mô hình.

## Mục tiêu

1. Kiểm tra môi trường Jupyter Notebook.
2. Đọc đúng bốn file NHANES gốc `.xpt`: `DEMO_L`, `BMX_L`, `GHB_L`, `DIQ_L`.
3. Xem số dòng/cột, `SEQN`, biến quan trọng, số lượng thiếu dữ liệu.
4. Ghép bảng bằng `SEQN` và xác định sơ bộ số mẫu có HbA1c ở người trưởng thành.
5. Lưu thống kê *do bạn chạy thực tế* vào `02-Data-Understanding.md`.

**Chưa làm ở bước này:** tạo nhãn cuối, xử lý missing values, chia train/test, fit preprocessing, train ML, đánh giá mô hình.

**Chú ý:** `LBXGH` là HbA1c (biến ứng viên để tạo nhãn về sau). Không đưa `LBXGH` vào feature dự đoán nếu nhãn được tạo từ chính nó. Nguồn NHANES này là khảo sát cắt ngang; chưa phải dữ liệu dự đoán mắc tiểu đường trong tương lai.

## 0. Chuẩn bị các file `.xpt` (thực hiện thủ công)

Từ website CDC, mở trang **codebook**, sau đó tải **Data File `.xpt`** tương ứng:

- [`DEMO_L.xpt` — Demographics](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.htm)
- [`BMX_L.xpt` — Body Measures](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.htm)
- [`GHB_L.xpt` — Glycohemoglobin / HbA1c](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/GHB_L.htm)
- [`DIQ_L.xpt` — Diabetes Questionnaire](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DIQ_L.htm)

Đặt đủ 4 file vào thư mục **`03-ML-Research/data/raw/`**. Giữ nguyên tên file gốc (chữ hoa/chữ thường đều được code tìm). Không chỉnh sửa file gốc.

Các file `BPXO_L`, `PAQ_L`, `SMQ_L`, `GLU_L` sẽ được bổ sung *sau khi* hoàn tất lượt kiểm tra đầu tiên; cần lưu ý quy tắc mẫu con và trọng số riêng của `GLU_L`.

**Cách tải chắc chắn nhất:** mở danh sách CDC theo nhóm: [Demographics](https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Demographics&Cycle=2021-2023), [Examination](https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Examination&Cycle=2021-2023), [Laboratory](https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Laboratory&Cycle=2021-2023), [Questionnaire](https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Questionnaire&Cycle=2021-2023). Tìm dòng mang tên tệp và bấm **Data [XPT]** (không bấm Doc nếu muốn lấy dữ liệu).

## 1. Kiểm tra kernel, thư mục đang làm việc

Chạy ô bên dưới. Trong Jupyter phải chọn **kernel Python của môi trường dự án**. Thư mục làm việc là nơi chứa notebook hoặc thư mục gốc `03-ML-Research`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

print('Python:', sys.version.split()[0])
print('Pandas:', pd.__version__)
print('Working directory:', Path.cwd())

# Notebook nằm trong notebooks/, nhưng người dùng đôi khi mở Jupyter từ thư mục gốc.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
print('Project root:', PROJECT_ROOT)
print('Data directory:', RAW_DIR)

Python: 3.13.9
Pandas: 3.0.0
Working directory: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\notebooks
Project root: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research
Data directory: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\raw


## 2. Kiểm tra bốn file dữ liệu gốc

Nếu báo `FileNotFoundError`, hãy xem đường dẫn in ở ô trước và chuyển các tệp vào đúng thư mục `data/raw`. Không nên bỏ qua lỗi hoặc tạo dữ liệu giả.

In [2]:
REQUIRED = ('DEMO_L', 'BMX_L', 'GHB_L', 'DIQ_L')

# Hỗ trợ cả đuôi .xpt và .XPT.
def locate_xpt(stem: str) -> Path:
    for ext in ('.xpt', '.XPT'):
        candidate = RAW_DIR / f'{stem}{ext}'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f'Thiếu {stem}.xpt trong {RAW_DIR}. Tải file gốc từ CDC rồi chạy lại.'
    )

paths = {name: locate_xpt(name) for name in REQUIRED}
for name, path in paths.items():
    print(f'{name}: {path.name} ({path.stat().st_size / 1024**2:.2f} MB)')

DEMO_L: DEMO_L.xpt (2.46 MB)
BMX_L: BMX_L.xpt (1.49 MB)
GHB_L: GHB_L.xpt (0.17 MB)
DIQ_L: DIQ_L.xpt (0.81 MB)


## 3. Đọc XPT bằng Pandas

`pd.read_sas(..., format='xport')` hỗ trợ file SAS Transport (XPT). Đây là bước **đọc dữ liệu**, không làm sạch hay thay đổi dữ liệu nguồn.

In [3]:
frames = {}
for name, path in paths.items():
    frames[name] = pd.read_sas(path, format='xport')
    df = frames[name]
    print(f'{name:7s}: {len(df):>7,} dòng × {df.shape[1]:>3} cột')

# Xem nhanh tối đa 5 bản ghi của DEMO_L:
display(frames['DEMO_L'].head())

DEMO_L :  11,933 dòng ×  27 cột
BMX_L  :   8,860 dòng ×  22 cột
GHB_L  :   7,199 dòng ×   3 cột
DIQ_L  :  11,744 dòng ×   9 cột


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHRGND,DMDHRAGZ,DMDHREDZ,DMDHRMAZ,DMDHSEDZ,WTINT2YR,WTMEC2YR,SDMVSTRA,SDMVPSU,INDFMPIR
0,130378.0,12.0,2.0,1.0,43.0,NaN,5.0,6.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,50055.450807,54374.463898,173.0,2.0,5.00
1,130379.0,12.0,2.0,1.0,66.0,NaN,3.0,3.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,29087.450605,34084.721548,173.0,2.0,5.00
2,130380.0,12.0,2.0,2.0,44.0,NaN,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,80062.674301,81196.277992,174.0,1.0,1.41
3,130381.0,12.0,2.0,2.0,5.0,NaN,5.0,7.0,1.0,71.0,...,2.0,2.0,2.0,3.0,NaN,38807.268902,55698.607106,182.0,2.0,1.53
4,130382.0,12.0,2.0,1.0,2.0,NaN,3.0,3.0,2.0,34.0,...,2.0,2.0,3.0,1.0,2.0,30607.519774,36434.146346,182.0,2.0,3.60


## 4. Kiểm tra khóa nối `SEQN`

Mỗi file nên có `SEQN` duy nhất theo người tham gia trong chu kỳ này. Nếu phát hiện bản ghi trùng hoặc không có khóa, cần dừng và kiểm tra nguồn file trước khi merge.

In [4]:
key_summary = []
for name, df in frames.items():
    if 'SEQN' not in df.columns:
        raise KeyError(f'{name} không có SEQN — kiểm tra lại đúng file CDC chưa.')
    key_summary.append({
        'file': name,
        'rows': len(df),
        'unique_seqn': df['SEQN'].nunique(dropna=True),
        'missing_seqn': int(df['SEQN'].isna().sum()),
        'duplicate_seqn': int(df['SEQN'].duplicated().sum())
    })
key_report = pd.DataFrame(key_summary)
display(key_report)

assert (key_report['duplicate_seqn'] == 0).all(), 'SEQN bị trùng, không merge vội.'
assert (key_report['missing_seqn'] == 0).all(), 'SEQN bị thiếu, không merge vội.'

,file,rows,unique_seqn,missing_seqn,duplicate_seqn
0,DEMO_L,11933,11933,0,0
1,BMX_L,8860,8860,0,0
2,GHB_L,7199,7199,0,0
3,DIQ_L,11744,11744,0,0


## 5. Đối chiếu biến quan trọng với codebook

Chúng ta chỉ thống kê các biến **đã xác minh** trên trang CDC; trường không có sẽ hiện rõ. Các biến ví dụ:

- `RIDAGEYR`: tuổi; `RIAGENDR`: giới (DEMO)
- `BMXBMI`: BMI (BMX)
- `LBXGH`: HbA1c %; `WTPH2YR`: trọng số phlebotomy (GHB)
- `DIQ010`: từng được bác sĩ thông báo có diabetes (DIQ)
- `RIDEXPRG`: tình trạng mang thai chỉ áp dụng cho nhóm đối tượng phù hợp, **không được coi NaN là chắc chắn không mang thai**.

In [5]:
want = {
    'DEMO_L': ['SEQN','RIDAGEYR','RIAGENDR','RIDEXPRG','WTMEC2YR','SDMVSTRA','SDMVPSU'],
    'BMX_L': ['SEQN','BMXBMI','BMXWAIST'],
    'GHB_L': ['SEQN','LBXGH','WTPH2YR'],
    'DIQ_L': ['SEQN','DIQ010','DIQ160'],
}

for name, cols in want.items():
    present = [c for c in cols if c in frames[name].columns]
    absent = [c for c in cols if c not in frames[name].columns]
    print(f'\n{name} — Có: {present}')
    if absent:
        print('  Chưa tìm thấy:', absent)


DEMO_L — Có: ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDEXPRG', 'WTMEC2YR', 'SDMVSTRA', 'SDMVPSU']

BMX_L — Có: ['SEQN', 'BMXBMI', 'BMXWAIST']

GHB_L — Có: ['SEQN', 'LBXGH', 'WTPH2YR']

DIQ_L — Có: ['SEQN', 'DIQ010', 'DIQ160']


## 6. Ghép thử 4 bảng bằng `SEQN` (giữ nguyên dân số DEMO)

Dùng **left join** từ DEMO để nhận biết dữ liệu thiếu ở các tệp con. `validate='one_to_one'` kiểm tra tính hợp lệ của phép ghép. Chỉ chọn cột liên quan để tránh lẫn mã trùng giữa các tệp.

**Chưa lọc nhãn, chưa xử lý giá trị thiếu, chưa huấn luyện.**

In [6]:
def only_existing(df, cols):
    return df[[col for col in cols if col in df.columns]].copy()

merged = only_existing(frames['DEMO_L'], want['DEMO_L'])
for name in ('BMX_L', 'GHB_L', 'DIQ_L'):
    extra = only_existing(frames[name], want[name])
    merged = merged.merge(extra, on='SEQN', how='left', validate='one_to_one')

print('DEMO_L gốc:', len(frames['DEMO_L']), 'dòng')
print('Sau ghép 4 bảng:', len(merged), 'dòng')
assert len(merged) == len(frames['DEMO_L']), 'Mất/tăng dòng khi ghép — cần kiểm tra.'
print('Số cột sau ghép:', merged.shape[1])
display(merged.head())

DEMO_L gốc: 11933 dòng
Sau ghép 4 bảng: 11933 dòng
Số cột sau ghép: 13


,SEQN,RIDAGEYR,RIAGENDR,RIDEXPRG,WTMEC2YR,SDMVSTRA,SDMVPSU,BMXBMI,BMXWAIST,LBXGH,WTPH2YR,DIQ010,DIQ160
0,130378.0,43.0,1.0,NaN,54374.463898,173.0,2.0,27.0,98.3,5.6,56042.129410,2.0,2.0
1,130379.0,66.0,1.0,NaN,34084.721548,173.0,2.0,33.5,114.7,5.6,37435.705647,2.0,2.0
2,130380.0,44.0,2.0,2.0,81196.277992,174.0,1.0,29.7,93.5,6.2,85328.844519,1.0,NaN
3,130381.0,5.0,2.0,NaN,55698.607106,182.0,2.0,23.8,70.4,NaN,NaN,2.0,NaN
4,130382.0,2.0,1.0,NaN,36434.146346,182.0,2.0,NaN,NaN,NaN,NaN,2.0,NaN


## 7. Kiểm tra số người trưởng thành và số người có HbA1c

Đây chỉ là bước kiểm tra khả năng sử dụng dữ liệu. **Chưa loại trừ thai kỳ, chưa loại người đã được chẩn đoán, chưa tạo nhãn.** Lấy nhóm từ 20 tuổi trở lên để phù hợp với phương án nghiên cứu sơ bộ của Bước 1.

In [7]:
if 'RIDAGEYR' not in merged.columns or 'LBXGH' not in merged.columns:
    raise KeyError('Thiếu RIDAGEYR hoặc LBXGH: hãy kiểm tra lại các file đã đọc.')

adults = merged.loc[merged['RIDAGEYR'] >= 20].copy()

print('Toàn bộ DEMO:', len(merged))
print('Người >= 20 tuổi:', len(adults))
print('Người >= 20 tuổi có LBXGH:', adults['LBXGH'].notna().sum())
print('Người >= 20 tuổi thiếu LBXGH:', adults['LBXGH'].isna().sum())

if 'DIQ010' in adults.columns:
    print('\nMã trả lời DIQ010 (chưa gán nhãn):')
    display(adults['DIQ010'].value_counts(dropna=False).sort_index())

print('\nMô tả HbA1c ở người >= 20 tuổi (chỉ thống kê):')
display(adults['LBXGH'].describe())

Toàn bộ DEMO: 11933
Người >= 20 tuổi: 7809
Người >= 20 tuổi có LBXGH: 5767
Người >= 20 tuổi thiếu LBXGH: 2042

Mã trả lời DIQ010 (chưa gán nhãn):


DIQ010
1.0    1070
2.0    6466
3.0     270
9.0       1
NaN       2
Name: count, dtype: int64


Mô tả HbA1c ở người >= 20 tuổi (chỉ thống kê):


count    5767.000000
mean        5.788174
std         1.112402
min         3.200000
25%         5.200000
50%         5.500000
75%         5.900000
max        17.100000
Name: LBXGH, dtype: float64

## 8. Kiểm tra dữ liệu thiếu theo trường

Chúng ta mới **quan sát** tỷ lệ thiếu; chưa thay thế NaN bằng 0 hoặc giá trị trung bình. Thiếu có thể do chưa đủ tuổi, không được hỏi, không tham gia xét nghiệm hoặc lý do khác. Quyết định tiền xử lý chỉ được đưa ra sau khi xem codebook và EDA.

In [8]:
fields = [c for c in ['RIDAGEYR','RIAGENDR','BMXBMI','BMXWAIST',
                      'LBXGH','WTPH2YR','DIQ010','DIQ160','RIDEXPRG']
          if c in adults.columns]
missing_report = pd.DataFrame({
    'missing_count': adults[fields].isna().sum(),
    'missing_pct': (adults[fields].isna().mean() * 100).round(2),
    'non_missing': adults[fields].notna().sum()
}).sort_values('missing_pct', ascending=False)
display(missing_report)

,missing_count,missing_pct,non_missing
RIDEXPRG,6306,80.75,1503
BMXWAIST,2047,26.21,5762
LBXGH,2042,26.15,5767
BMXBMI,1839,23.55,5970
WTPH2YR,1745,22.35,6064
DIQ160,1344,17.21,6465
DIQ010,2,0.03,7807
RIDAGEYR,0,0.00,7809
RIAGENDR,0,0.00,7809


## 9. Tổng hợp thông tin để gửi lại người hướng dẫn

Chạy ô cuối cùng và **sao chép phần kết quả tổng hợp**, không cần gửi file dữ liệu `.xpt` hoặc dữ liệu nhạy cảm. Tôi sẽ sử dụng con số *thực tế do bạn chạy* để cập nhật `02-Data-Understanding.md` và quyết định bước EDA tiếp theo.

In [9]:
print('===== BƯỚC 2 — NHANES DATA UNDERSTANDING =====')
print('Python:', sys.version.split()[0], '| Pandas:', pd.__version__)
for name in REQUIRED:
    print(f'{name}: {frames[name].shape[0]} dòng, {frames[name].shape[1]} cột')
print('SEQN trùng (tổng):', int(key_report['duplicate_seqn'].sum()))
print('Merged:', merged.shape)
print('Adults >=20:', len(adults))
print('Adults >=20 có HbA1c:', int(adults['LBXGH'].notna().sum()))
print('Adults >=20 thiếu HbA1c:', int(adults['LBXGH'].isna().sum()))
print('Missing % cột chính:')
print(missing_report['missing_pct'].to_string())
print('===== KẾT THÚC =====')

===== BƯỚC 2 — NHANES DATA UNDERSTANDING =====
Python: 3.13.9 | Pandas: 3.0.0
DEMO_L: 11933 dòng, 27 cột
BMX_L: 8860 dòng, 22 cột
GHB_L: 7199 dòng, 3 cột
DIQ_L: 11744 dòng, 9 cột
SEQN trùng (tổng): 0
Merged: (11933, 13)
Adults >=20: 7809
Adults >=20 có HbA1c: 5767
Adults >=20 thiếu HbA1c: 2042
Missing % cột chính:
RIDEXPRG    80.75
BMXWAIST    26.21
LBXGH       26.15
BMXBMI      23.55
WTPH2YR     22.35
DIQ160      17.21
DIQ010       0.03
RIDAGEYR     0.00
RIAGENDR     0.00
===== KẾT THÚC =====


## Đến đây hãy dừng lại

Bạn gửi cho tôi ảnh hoặc văn bản của output từ **ô số 9** và các lỗi nếu có. Chúng ta sẽ xác nhận dữ liệu đã đọc đúng rồi mới bổ sung `BPXO_L`, `PAQ_L`, `SMQ_L`, `GLU_L` và chuẩn bị bước 3 (EDA).

Không thực hiện `train_test_split`, `fit`, `predict`, tạo label hay xuất mô hình ở notebook này.

### Tài liệu nguồn CDC
- [NHANES 2021–2023](https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?Cycle=2021-2023)
- [GHB_L (LBXGH và WTPH2YR)](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/GHB_L.htm)
- [GLU_L (LBXGLU và WTSAF2YR)](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/GLU_L.htm)
- [NHANES analytic guidance / weighting](https://wwwn.cdc.gov/nchs/nhanes/tutorials/weighting.aspx)